# LeadFormer: Scalp EEG → iEEG (cross-subject), chạy trên Google Colab

Chạy **tuần tự từ trên xuống**. Mọi kết quả (code, dữ liệu đã tiền xử lý, checkpoint, log, hình) được lưu ở Google Drive: `MyDrive/iEEG_S2I/`.

| Bước | Nội dung | Thời gian ước tính (T4) |
|---|---|---|
| 1–3 | Mount Drive, kiểm tra GPU, cài thư viện | 2 phút |
| 4 | Tải ds004752 từ OpenNeuro (S3) về ổ local của Colab | 5–15 phút |
| 5 | Ghi code `s2i/` lên Drive | vài giây |
| 6 | Smoke test trên dữ liệu giả | 1–2 phút |
| 7–8 | Tiền xử lý và xem tổng quan dữ liệu | 20–40 phút |
| 9 | **Kiểm định tham chiếu chung** (chạy trước khi tin bất kỳ con số nào) | 2–5 phút |
| 10 | LOSO chính (bipolar + CAR) | vài giờ, **tự resume** |
| 11 | LOSO đối chứng (monopolar, để so với CAST) | tuỳ chọn |
| 12–14 | Bảng kết quả, biểu đồ, waveform | vài phút |

> **Nếu Colab bị ngắt kết nối:** chạy lại cell 1 và 3, rồi chạy lại cell đang dở. Tiền xử lý bỏ qua các subject đã xong, training resume theo **từng epoch** trong mỗi fold.
> Dữ liệu thô (~12 GB) chỉ nằm trên ổ local của Colab và mất khi ngắt kết nối. Bản đã tiền xử lý (~1–2 GB) nằm trên Drive, nên sau khi bước 7 xong thì **không cần tải lại**.

In [ ]:
#@title 1. Mount Google Drive + cấu hình đường dẫn
from google.colab import drive
drive.mount('/content/drive')

import os, sys
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
MODE     = 'full'                              #@param ["fast", "full"]
RAW_DIR  = '/content/ds004752'                 # dữ liệu thô: ổ local Colab (nhanh)
CODE_DIR = f'{ROOT}/code'
DATA_DIR = f'{ROOT}/data/s2i_v1'
RUNS_DIR = f'{ROOT}/runs'
for p in (f'{CODE_DIR}/s2i/data', f'{CODE_DIR}/s2i/model', DATA_DIR, RUNS_DIR):
    os.makedirs(p, exist_ok=True)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
print('ROOT =', ROOT, '| MODE =', MODE)

In [ ]:
#@title 2. Kiểm tra GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'

In [ ]:
#@title 3. Cài thư viện (chạy lại sau mỗi lần reconnect)
!pip -q install mne awscli
import mne; print('mne', mne.__version__)

## 4. Tải dữ liệu OpenNeuro ds004752
Dùng bucket S3 công khai của OpenNeuro (không cần tài khoản). `aws s3 sync` chạy lại được nhiều lần (chỉ tải phần còn thiếu).
Bỏ qua thư mục `derivatives/` (beamforming) vì không dùng đến.

Nếu **đã có** `DATA_DIR` với đủ file `.pt` từ lần trước thì bỏ qua cell này và cell 7.

In [ ]:
#@title 4. Tải ds004752 (~12 GB) về /content
import shutil, glob
print(f"Dung lượng trống /content: {shutil.disk_usage('/content').free/1e9:.1f} GB")
already = sorted(glob.glob(f'{DATA_DIR}/sub-*.pt'))
if len(already) >= 14:
    print(f'Đã có {len(already)} subject tiền xử lý trên Drive -> bỏ qua tải.')
else:
    !aws s3 sync --no-sign-request s3://openneuro.org/ds004752 {RAW_DIR} --exclude "derivatives/*" --only-show-errors
    !du -sh {RAW_DIR}
    n_eeg  = len(glob.glob(f'{RAW_DIR}/sub-*/ses-*/eeg/*.edf'))
    n_ieeg = len(glob.glob(f'{RAW_DIR}/sub-*/ses-*/ieeg/*.edf'))
    n_el   = len(glob.glob(f'{RAW_DIR}/sub-*/ses-*/ieeg/*electrodes.tsv'))
    print(f'EEG edf: {n_eeg} | iEEG edf: {n_ieeg} | electrodes.tsv: {n_el}')
    print(sorted(os.listdir(RAW_DIR))[:20])

## 5. Ghi code `s2i/` lên Drive
Các cell dưới đây ghi từng file của package lên `MyDrive/iEEG_S2I/code/s2i/`. Chỉ cần chạy **một lần** (code nằm lại trên Drive). Muốn sửa code thì sửa trực tiếp file trên Drive hoặc sửa cell rồi chạy lại.

In [ ]:
%%writefile {CODE_DIR}/s2i/__init__.py
"""
s2i — Scalp-to-Intracranial translation (rebuilt from scratch).

Package layout
--------------
s2i/montage.py                 10-20 channel canonicalisation + 3D positions
s2i/data/preprocess_ds004752.py BIDS -> per-subject trial-level .pt files
s2i/data/dataset.py            trial-level storage, leakage-free splits, windowing, collate
s2i/model/leadformer.py        LeadFormer: geometry-aware encoder-decoder
s2i/losses.py                  NLL (observability) + Pearson + multi-resolution STFT
s2i/metrics.py                 per-contact r / R^2 (unclipped) / band-limited r
s2i/baselines.py               lagged ridge regression baseline (must be beaten)
s2i/train_loso.py              full LOSO protocol: pretrain -> zero-shot -> calibration -> ridge
s2i/diagnose_reference.py      common-reference contamination test
s2i/smoke_test.py              synthetic end-to-end test (no real data needed)
"""

In [ ]:
%%writefile {CODE_DIR}/s2i/montage.py
"""
10-20 scalp channel canonicalisation.

Old pipeline bug: EEG channels were zero-padded *by index* to 21. Subjects have
different montages (6, 8, 17, 18, 19, 21 ch) and different orderings
(e.g. sub-05 starts with Fp1, sub-01 with F3), so input slot i meant a different
electrode for each subject -> the "universal" encoder could not be universal.

Here every scalp channel is identified by its canonical 10-20 name and a 3D
position, and the model treats channels as a *set* of tokens with positional
embeddings, so any montage works.
"""
import re

# Old 10-20 nomenclature -> modern (MNE standard_1020 uses modern names)
_ALIASES = {"T3": "T7", "T4": "T8", "T5": "P7", "T6": "P8"}

# Non-EEG / reference channels to drop
NON_EEG_TOKENS = ("EKG", "ECG", "EMG", "EOG", "A1", "A2", "M1", "M2",
                  "STATUS", "TRIG", "PHOTIC", "RESP", "SPO2", "PULSE")


def canonical_eeg_name(raw_name: str):
    """'EEG F3-Ref' -> 'F3', 'T3' -> 'T7', 'FP1' -> 'Fp1'. Returns None if not EEG."""
    n = raw_name.strip()
    n = re.sub(r"^(EEG|eeg)[\s_:-]*", "", n)
    n = re.split(r"[-\s/]", n)[0]          # drop reference suffix
    if not n:
        return None
    up = n.upper()
    if any(tok == up or up.startswith(tok) for tok in NON_EEG_TOKENS):
        return None
    # restore canonical casing: Fp1, Fz, Cz, AF3, FC1 ...
    n = "Fp" + up[2:] if up.startswith("FP") else up
    if n.endswith("Z"):
        n = n[:-1] + "z"
    return _ALIASES.get(n, n)


def standard_positions_mm():
    """Canonical-name -> (x, y, z) in mm (MNE standard_1020, head frame).

    Head frame is close to (not identical to) MNI; the model learns a relative
    geometry bias, so a constant offset between frames is absorbed in training.
    """
    import mne
    mont = mne.channels.make_standard_montage("standard_1020")
    pos = mont.get_positions()["ch_pos"]
    out = {}
    for name, xyz in pos.items():
        c = canonical_eeg_name(name)
        if c is not None:
            out[c] = [float(v) * 1000.0 for v in xyz]
    return out

In [ ]:
%%writefile {CODE_DIR}/s2i/data/__init__.py
"""Data subpackage: preprocessing + dataset utilities."""

In [ ]:
%%writefile {CODE_DIR}/s2i/data/preprocess_ds004752.py
"""
Rebuilt preprocessing for OpenNeuro ds004752 (trial-level, leakage-free, reference-aware).

What changed vs. the old pipelines (data_pipeline.py / reprocess_ds004752.py)
---------------------------------------------------------------------------
1. Scalp channels are canonicalised to 10-20 names and given 3D positions
   (old: zero-padded by index -> channel i meant different electrodes per subject).
2. Two scalp references are stored: original amplifier reference ('eeg_orig') and
   common-average reference ('eeg_car').
3. Two intracranial targets are stored: monopolar ('ieeg_mono') and bipolar
   ('ieeg_bip', adjacent contacts on the same shaft). Monopolar contacts share the
   recording reference; any signal on that reference is trivially "predictable"
   from scalp channels recorded against the same reference (see diagnose_reference.py).
4. Data are stored per TRIAL as continuous segments. Splits are done by trial,
   chronologically (old: random split of 50%-overlapping 2 s windows -> neighbouring
   windows shared 1 s of samples between calibration and test).
5. Normalisation is robust (median / MAD) per channel per session, NOT per window.
6. Band 0.5-70 Hz (avoids 100 Hz harmonic region after 200 Hz resampling).

Usage
-----
python -m s2i.data.preprocess_ds004752 --bids c:\\iEEG_EEG\\data\\ds004752 --out c:\\iEEG_EEG\\data\\s2i_v1
"""
import argparse
import datetime
import gc
import glob
import json
import os
import re
import shutil
import sys
import tempfile

import numpy as np
import pandas as pd
import torch

from s2i.montage import canonical_eeg_name, standard_positions_mm

FS = 200
BAND = (0.5, 70.0)
NOTCH = (50.0, 100.0)
EEG_REJECT_Z = 15.0      # gross scalp artifact -> drop trial
IEEG_CLIP_Z = 10.0       # clip epileptiform outliers in targets (do not drop trials)


# ----------------------------------------------------------------------------- anatomy
def region_category(r: str) -> str:
    r = str(r).upper()
    table = [
        ("Hippocampus", ["HIPP"]), ("Amygdala", ["AMYG"]),
        ("Parahippocampal", ["PHG", "PARAHIPP"]),
        ("STG", ["STG", "SUPERIOR TEMPORAL GYRUS"]), ("MTG", ["MTG", "MIDDLE TEMPORAL GYRUS"]),
        ("ITG", ["ITG", "INFERIOR TEMPORAL GYRUS"]), ("Fusiform", ["FUG", "FUSIFORM"]),
        ("IPL", ["IPL", "INFERIOR PARIETAL"]), ("Postcentral", ["POG", "POSTCENTRAL"]),
        ("Precentral", ["PRG", "PRECENTRAL"]), ("Insula", ["INS", "INSUL"]),
        ("Occipital", ["LOCC", "OCCIPITAL"]), ("pSTS", ["PSTS"]),
        ("Frontal", ["SFG", "MFG", "IFG", "ORG", "FRONTAL"]),
        ("Basal Ganglia", ["BG,", "BASAL", "PUTAMEN"]),
    ]
    for cat, keys in table:
        if any(k in r for k in keys):
            return cat
    return "Unlabeled"


def depth_of(cat: str) -> str:
    if cat in {"Hippocampus", "Amygdala", "Basal Ganglia"}:
        return "Deep"
    if cat in {"Parahippocampal", "Insula", "Fusiform"}:
        return "Mid"
    if cat == "Unlabeled":
        return "Unknown"
    return "Cortical"


def make_bipolar(names):
    """Adjacent contacts (number differs by exactly 1) on the same shaft."""
    shafts = {}
    for idx, n in enumerate(names):
        m = re.match(r"^(.*?)(\d+)$", n)
        if not m:
            continue
        shafts.setdefault(m.group(1), []).append((int(m.group(2)), idx))
    pairs = []
    for pref in sorted(shafts):
        cs = sorted(shafts[pref])
        for (n0, i0), (n1, i1) in zip(cs[:-1], cs[1:]):
            if n1 - n0 == 1:
                pairs.append((i1, i0))       # deeper-numbered minus shallower-numbered
    return pairs


# ----------------------------------------------------------------------------- EDF I/O
def _valid_edf_field(s, fmt):
    try:
        datetime.datetime.strptime(s, fmt)
        return True
    except ValueError:
        return False


def _patch_edf_header(src, dst):
    """Copy src -> dst and replace an invalid startdate/starttime header field.

    EDF header bytes 168-175 = 'dd.mm.yy', 176-183 = 'hh.mm.ss'. Some ds004752 files
    contain e.g. seconds >= 60 there, which MNE rejects. The recording start time is
    not used by this pipeline, so invalid fields are replaced by a neutral value.
    Returns a description of what was patched ('' if nothing was invalid).
    """
    shutil.copyfile(src, dst)
    fixed = []
    with open(dst, "r+b") as f:
        f.seek(168)
        hdr = f.read(16).decode("ascii", "replace")
        date, time = hdr[:8], hdr[8:16]
        if not _valid_edf_field(date, "%d.%m.%y"):
            fixed.append(f"date '{date}'")
            date = "01.01.85"
        if not _valid_edf_field(time, "%H.%M.%S"):
            fixed.append(f"time '{time}'")
            time = "00.00.00"
        f.seek(168)
        f.write((date + time).encode("ascii"))
    return ", ".join(fixed)


def read_edf(path):
    """mne.io.read_raw_edf with a fallback for corrupted start date/time headers."""
    import mne
    try:
        return mne.io.read_raw_edf(path, preload=True, verbose=False)
    except ValueError as e:
        tmp = os.path.join(tempfile.gettempdir(), "s2i_hdrfix_" + os.path.basename(path))
        try:
            fixed = _patch_edf_header(path, tmp)
            if not fixed:
                raise
            print(f"  [WARN] {os.path.basename(path)}: invalid EDF header ({e}); "
                  f"patched {fixed} in a temp copy", flush=True)
            return mne.io.read_raw_edf(tmp, preload=True, verbose=False)
        finally:
            if os.path.exists(tmp):
                os.remove(tmp)


# ----------------------------------------------------------------------------- signal
def filt_resample(raw):
    nyq = raw.info["sfreq"] / 2.0
    notch = [f for f in NOTCH if f < nyq - 1]
    if notch:
        raw.notch_filter(notch, picks="all", verbose=False)
    raw.filter(BAND[0], min(BAND[1], nyq - 1.0), picks="all", verbose=False)
    if raw.info["sfreq"] != FS:
        raw.resample(FS, verbose=False)
    return raw.get_data(picks="all")


def robust_norm(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    mad = np.where(mad < 1e-12, 1.0, mad)
    return (x - med) / mad


def common_mode_stats(mono_v, eeg_v):
    """Diagnostics computed on raw-referenced volts (before any re-referencing)."""
    sub = slice(None, None, 4)
    X = mono_v[:, sub]
    C = np.corrcoef(X)
    C = np.nan_to_num(C)
    pc1 = float(np.linalg.eigvalsh(C)[-1] / C.shape[0])
    m_i = X.mean(0)
    rs = [abs(np.corrcoef(m_i, e[sub])[0, 1]) for e in eeg_v]
    return {"mono_pc1_ratio": pc1, "mean_abs_corr_iEEGmean_vs_EEG": float(np.nanmean(rs)),
            "max_abs_corr_iEEGmean_vs_EEG": float(np.nanmax(rs))}


# ----------------------------------------------------------------------------- subject
def process_subject(bids, sub, pos_lut):
    import mne
    mne.set_log_level("ERROR")
    sessions = sorted(d for d in os.listdir(os.path.join(bids, sub)) if d.startswith("ses-"))
    contact_names = contact_mni = contact_region = None
    per_session = []

    for ses in sessions:
        base = os.path.join(bids, sub, ses)
        ef = glob.glob(os.path.join(base, "eeg", "*.edf"))
        jf = glob.glob(os.path.join(base, "ieeg", "*.edf"))
        if not ef or not jf:
            continue
        try:
            raw_e = read_edf(ef[0])
            raw_i = read_edf(jf[0])
        except Exception as e:  # one unreadable session must not kill the whole run
            print(f"  [{ses}] cannot read EDF ({type(e).__name__}: {e}), skip", flush=True)
            continue

        # ---- scalp channels -> canonical names with known positions
        keep, canon = [], []
        for ch in raw_e.ch_names:
            c = canonical_eeg_name(ch)
            if c is not None and c in pos_lut and c not in canon:
                keep.append(ch)
                canon.append(c)
        if len(keep) < 4:
            print(f"  [{ses}] <4 usable scalp channels, skip")
            continue
        raw_e.pick(keep)
        raw_e.reorder_channels(keep)

        # ---- intracranial contacts (fixed from the first usable session)
        if contact_names is None:
            tf = glob.glob(os.path.join(base, "ieeg", "*electrodes.tsv"))
            if not tf:
                continue
            el = pd.read_csv(tf[0], sep="\t")
            names, mni, reg = [], [], []
            for _, row in el.iterrows():
                xyz = [row.get("x"), row.get("y"), row.get("z")]
                if row["name"] in raw_i.ch_names and np.all(np.isfinite(np.asarray(xyz, float))):
                    names.append(row["name"])
                    mni.append([float(v) for v in xyz])
                    reg.append(str(row.get("AnatomicalLocation", "unknown")))
            if not names:
                continue
            contact_names, contact_mni, contact_region = names, np.asarray(mni, np.float32), reg
        if any(c not in raw_i.ch_names for c in contact_names):
            print(f"  [{ses}] contact set differs from first session, skip")
            continue
        raw_i.pick(contact_names)
        raw_i.reorder_channels(contact_names)

        e_v = filt_resample(raw_e)
        i_v = filt_resample(raw_i)
        n = min(e_v.shape[1], i_v.shape[1])
        e_v, i_v = e_v[:, :n], i_v[:, :n]

        evf = glob.glob(os.path.join(base, "ieeg", "*events.tsv")) or \
            glob.glob(os.path.join(base, "eeg", "*events.tsv"))
        if not evf:
            continue
        events = pd.read_csv(evf[0], sep="\t").sort_values("onset")
        per_session.append(dict(ses=ses, eeg_names=canon, e_v=e_v, i_v=i_v, events=events))
        print(f"  [{ses}] scalp {len(canon)} ch, iEEG {len(contact_names)} ch, {n / FS:.0f}s, "
              f"{len(events)} events")

    if not per_session:
        return None

    # ---- scalp channel set common to all sessions
    common = [c for c in per_session[0]["eeg_names"]
              if all(c in s["eeg_names"] for s in per_session)]
    pairs = make_bipolar(contact_names)
    if not pairs:
        print("  [WARN] no bipolar pairs")

    bufs = {k: [] for k in ("eeg_orig", "eeg_car", "ieeg_mono", "ieeg_bip")}
    bounds, t_ses, t_id, diags = [], [], [], []
    cursor, n_rej = 0, 0
    for s in per_session:
        idx = [s["eeg_names"].index(c) for c in common]
        e_v = s["e_v"][idx]
        i_v = s["i_v"]
        diags.append(common_mode_stats(i_v, e_v))
        e_orig = robust_norm(e_v)
        e_car = robust_norm(e_v - e_v.mean(0, keepdims=True))
        mono = np.clip(robust_norm(i_v), -IEEG_CLIP_Z, IEEG_CLIP_Z)
        bip_v = np.stack([i_v[a] - i_v[b] for a, b in pairs]) if pairs else np.zeros((0, i_v.shape[1]))
        bip = np.clip(robust_norm(bip_v), -IEEG_CLIP_Z, IEEG_CLIP_Z) if pairs else bip_v

        for _, tr in s["events"].iterrows():
            art = tr.get("Artifact", 0)
            if pd.notna(art) and int(art) == 1:
                continue
            s0 = int(round(tr["onset"] * FS))
            s1 = int(round((tr["onset"] + tr["duration"]) * FS))
            if s0 < 0 or s1 > e_v.shape[1] or s1 - s0 < 2 * FS:
                continue
            if np.abs(e_car[:, s0:s1]).max() > EEG_REJECT_Z:
                n_rej += 1
                continue
            bufs["eeg_orig"].append(e_orig[:, s0:s1])
            bufs["eeg_car"].append(e_car[:, s0:s1])
            bufs["ieeg_mono"].append(mono[:, s0:s1])
            bufs["ieeg_bip"].append(bip[:, s0:s1])
            bounds.append((cursor, cursor + (s1 - s0)))
            cursor += s1 - s0
            t_ses.append(s["ses"])
            nt = tr.get("nTrial", np.nan)
            t_id.append(int(nt) if pd.notna(nt) else len(t_id))

    if not bounds:
        return None

    c_reg = contact_region
    c_cat = [region_category(r) for r in c_reg]
    b_reg = [c_reg[a] if c_reg[a] == c_reg[b] else f"{c_reg[a]} | {c_reg[b]}" for a, b in pairs]
    b_cat = [c_cat[a] if c_cat[a] == c_cat[b] else "Mixed" for a, b in pairs]

    out = {
        "subject": sub, "fs": FS,
        "eeg_names": common,
        "eeg_pos": torch.tensor([pos_lut[c] for c in common], dtype=torch.float32),
        "contact_names": contact_names,
        "contact_mni": torch.from_numpy(contact_mni),
        "contact_region": c_reg, "contact_cat": c_cat,
        "contact_depth": [depth_of(c) for c in c_cat],
        "bip_pairs": pairs,
        "bip_names": [f"{contact_names[a]}-{contact_names[b]}" for a, b in pairs],
        "bip_mni": torch.from_numpy(
            np.stack([(contact_mni[a] + contact_mni[b]) / 2 for a, b in pairs]).astype(np.float32)
            if pairs else np.zeros((0, 3), np.float32)),
        "bip_region": b_reg, "bip_cat": b_cat,
        "bip_depth": [depth_of(c) if c != "Mixed" else "Mixed" for c in b_cat],
        "trial_bounds": torch.tensor(bounds, dtype=torch.long),
        "trial_session": t_ses, "trial_id": t_id,
        "diagnostics": {k: float(np.mean([d[k] for d in diags])) for k in diags[0]},
        "n_rejected_trials": n_rej,
    }
    for k, v in bufs.items():
        out[k] = torch.from_numpy(np.concatenate(v, axis=1).astype(np.float16))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--bids", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--subjects", nargs="*", default=None)
    ap.add_argument("--overwrite", action="store_true")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    pos_lut = standard_positions_mm()
    subs = a.subjects or sorted(d for d in os.listdir(a.bids) if d.startswith("sub-"))
    summary = []
    for sub in subs:
        print(f"\n=== {sub} ===", flush=True)
        path = os.path.join(a.out, f"{sub}.pt")
        if os.path.exists(path) and not a.overwrite:
            d = torch.load(path, weights_only=False)
            print("  already processed, skip")
        else:
            d = process_subject(a.bids, sub, pos_lut)
            if d is None:
                print("  no data")
                continue
            torch.save(d, path + ".tmp")
            os.replace(path + ".tmp", path)
        n_tr = len(d["trial_bounds"])
        dur = d["eeg_car"].shape[1] / FS
        info = {"subject": sub, "trials": n_tr, "minutes": round(dur / 60, 1),
                "eeg": d["eeg_names"], "contacts": len(d["contact_names"]),
                "bipolar": len(d["bip_names"]), "diagnostics": d["diagnostics"]}
        summary.append(info)
        print(f"  {n_tr} trials, {dur / 60:.1f} min, {len(d['eeg_names'])} scalp, "
              f"{len(d['contact_names'])} mono, {len(d['bip_names'])} bipolar | diag={d['diagnostics']}",
              flush=True)
        del d
        gc.collect()
    with open(os.path.join(a.out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/data/dataset.py
"""
Trial-level dataset utilities.

* Every split is done on whole trials, chronologically, with a gap -> no sample
  can appear in both calibration and test (old code split 50%-overlapping windows
  at random, so ~1/3 of test windows shared 1 s of data with a calibration window).
* Windows are cropped on the fly from continuous trial segments.
* Scalp channels and target contacts are variable-size sets -> padded + masked.
"""
import numpy as np
import torch
from torch.utils.data import Dataset


class Subject:
    """In-memory view of one preprocessed subject (float16 storage)."""

    def __init__(self, path, target="bip", eeg_ref="car"):
        d = torch.load(path, weights_only=False)
        self.sid = d["subject"]
        self.eeg = d["eeg_car" if eeg_ref == "car" else "eeg_orig"].half()
        self.eeg_pos = d["eeg_pos"].float()
        self.eeg_names = d["eeg_names"]
        key = "bip" if target == "bip" else "contact"
        self.tgt = d["ieeg_bip" if target == "bip" else "ieeg_mono"].half()
        self.tgt_pos = d[f"{key}_mni"].float()
        self.tgt_names = d[f"{key}_names"]
        self.tgt_region = d[f"{key}_region"]
        self.tgt_depth = d[f"{key}_depth"]
        self.bounds = d["trial_bounds"].numpy()
        self.diagnostics = d.get("diagnostics", {})

    @property
    def n_trials(self):
        return len(self.bounds)

    @property
    def n_targets(self):
        return self.tgt.shape[0]

    def trial(self, i):
        s0, s1 = self.bounds[i]
        return self.eeg[:, s0:s1], self.tgt[:, s0:s1]

    def trial_len(self, i):
        s0, s1 = self.bounds[i]
        return int(s1 - s0)


def chrono_split(n_trials, first_frac, gap=1):
    """First `first_frac` trials (chronological) vs. the rest after `gap` trials."""
    n_a = max(1, int(round(n_trials * first_frac)))
    a = list(range(0, n_a))
    b = list(range(min(n_trials, n_a + gap), n_trials))
    return a, b


class WindowDataset(Dataset):
    """
    mode='train': each item = (subject, trial); a random crop is drawn on access,
                  and at most k_max target contacts are randomly sub-sampled.
    mode='eval' : deterministic windows with `stride`, all contacts.
    """

    def __init__(self, subjects, trials, win, mode="train", stride=None,
                 k_max=None, crops_per_trial=4, seed=0):
        self.subjects, self.win, self.mode, self.k_max = subjects, win, mode, k_max
        self.rng = np.random.default_rng(seed)
        self.items = []
        for si, s in enumerate(subjects):
            for ti in trials[s.sid]:
                L = s.trial_len(ti)
                if L < win:
                    continue
                if mode == "train":
                    self.items += [(si, ti, -1)] * crops_per_trial
                else:
                    st = stride or win
                    for off in range(0, L - win + 1, st):
                        self.items.append((si, ti, off))

    def subject_balanced_weights(self):
        counts = {}
        for si, _, _ in self.items:
            counts[si] = counts.get(si, 0) + 1
        return torch.tensor([1.0 / counts[si] for si, _, _ in self.items], dtype=torch.double)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        si, ti, off = self.items[idx]
        s = self.subjects[si]
        e, y = s.trial(ti)
        if off < 0:
            off = int(self.rng.integers(0, e.shape[1] - self.win + 1))
        e = e[:, off:off + self.win].float()
        y = y[:, off:off + self.win].float()
        cidx = torch.arange(y.shape[0])
        if self.mode == "train" and self.k_max and y.shape[0] > self.k_max:
            cidx = torch.from_numpy(self.rng.choice(y.shape[0], self.k_max, replace=False))
            y = y[cidx]
        return {"eeg": e, "eeg_pos": s.eeg_pos, "tgt": y, "tgt_pos": s.tgt_pos[cidx],
                "cidx": cidx, "subj": si}


def collate(batch):
    B = len(batch)
    C = max(b["eeg"].shape[0] for b in batch)
    K = max(b["tgt"].shape[0] for b in batch)
    L = batch[0]["eeg"].shape[1]
    eeg = torch.zeros(B, C, L)
    eeg_pos = torch.zeros(B, C, 3)
    eeg_mask = torch.zeros(B, C, dtype=torch.bool)
    tgt = torch.zeros(B, K, L)
    tgt_pos = torch.zeros(B, K, 3)
    tgt_mask = torch.zeros(B, K, dtype=torch.bool)
    cidx = torch.zeros(B, K, dtype=torch.long)
    for i, b in enumerate(batch):
        c, k = b["eeg"].shape[0], b["tgt"].shape[0]
        eeg[i, :c], eeg_pos[i, :c], eeg_mask[i, :c] = b["eeg"], b["eeg_pos"], True
        tgt[i, :k], tgt_pos[i, :k], tgt_mask[i, :k] = b["tgt"], b["tgt_pos"], True
        cidx[i, :k] = b["cidx"]
    return {"eeg": eeg, "eeg_pos": eeg_pos, "eeg_mask": eeg_mask, "tgt": tgt,
            "tgt_pos": tgt_pos, "tgt_mask": tgt_mask, "cidx": cidx,
            "subj": torch.tensor([b["subj"] for b in batch])}

In [ ]:
%%writefile {CODE_DIR}/s2i/model/__init__.py
"""Model subpackage."""
from s2i.model.leadformer import LeadFormer, SubjectAdapter  # noqa: F401

In [ ]:
%%writefile {CODE_DIR}/s2i/model/leadformer.py
"""
LeadFormer — geometry-aware encoder-decoder for scalp-EEG -> iEEG translation.

Design goals (each one targets a concrete weakness of CAST):

1. Montage-agnostic encoder.
   CAST flattened (21 channels x patch) into one vector, so channel identity was
   tied to an array index that differed between subjects. LeadFormer tokenises
   every scalp channel separately (shared multi-scale conv tokenizer) and adds a
   3D positional embedding of the electrode. Attention alternates between the
   spatial axis (across electrodes, padded electrodes masked) and the temporal
   axis. Works for 6..21 electrodes without zero-padding hacks.

2. Subject-transferable decoder.
   CAST's decoder used `nn.Embedding(channel_index)` -> re-initialised from scratch
   for every new patient, so nothing about the output side transferred. Here every
   target contact is a query built from its MNI coordinate, so the *same* decoder
   is trained jointly on all training subjects and can be applied zero-shot to an
   unseen patient's electrode layout.

3. Lead-field-inspired attention prior.
   Volume conduction is instantaneous and decays with distance. Decoder queries
   cross-attend to scalp tokens at the *same* time step with an additive bias
   b_h(k, c) = MLP(r_kc) - softplus(a_h) * |r_kc|, where r_kc is the vector from
   contact k to electrode c. Different heads learn different spatial scales.

4. Observability head.
   A per-contact log-variance is predicted (heteroscedastic Gaussian NLL). It
   (a) down-weights unpredictable contacts during training, and (b) gives a
   principled, ground-truth-free ranking of reconstructable contacts, replacing
   the post-hoc "r >= 0.15 on calibration" threshold.

5. Light per-subject calibration.
   Optional `SubjectAdapter`: per-contact query offset + output gain/bias. Only
   these (and optionally the decoder) are tuned on a few minutes of data.
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def sinusoid_table(n, d, device=None):
    pos = torch.arange(n, device=device, dtype=torch.float32)[:, None]
    div = torch.exp(torch.arange(0, d, 2, device=device, dtype=torch.float32) * (-math.log(10000.0) / d))
    pe = torch.zeros(n, d, device=device)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class FourierXYZ(nn.Module):
    """Random-Fourier-feature embedding of 3D coordinates (mm)."""

    def __init__(self, d, n_freq=32, scale_mm=80.0, sigma=1.0, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.register_buffer("B", torch.randn(3, n_freq, generator=g) * sigma)
        self.scale = scale_mm
        self.mlp = nn.Sequential(nn.Linear(2 * n_freq + 3, d), nn.GELU(), nn.Linear(d, d))

    def forward(self, xyz):
        x = xyz / self.scale
        proj = 2 * math.pi * x @ self.B
        return self.mlp(torch.cat([proj.sin(), proj.cos(), x], -1))


class MHA(nn.Module):
    """Multi-head attention on (N, L, d) with additive bias and key-padding mask."""

    def __init__(self, d, h, dropout=0.0):
        super().__init__()
        assert d % h == 0
        self.h, self.dh, self.p = h, d // h, dropout
        self.q = nn.Linear(d, d)
        self.kv = nn.Linear(d, 2 * d)
        self.o = nn.Linear(d, d)

    def forward(self, x, ctx=None, bias=None, key_mask=None):
        ctx = x if ctx is None else ctx
        N, Lq, d = x.shape
        Lk = ctx.shape[1]
        q = self.q(x).view(N, Lq, self.h, self.dh).transpose(1, 2)
        k, v = self.kv(ctx).view(N, Lk, 2, self.h, self.dh).permute(2, 0, 3, 1, 4)
        mask = None
        if bias is not None:
            mask = bias.to(q.dtype)
        if key_mask is not None:
            km = torch.zeros(N, 1, 1, Lk, dtype=q.dtype, device=x.device)
            km = km.masked_fill(~key_mask[:, None, None, :], -1e4)
            mask = km if mask is None else mask + km
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                             dropout_p=self.p if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(N, Lq, d))


class FFN(nn.Module):
    def __init__(self, d, mult=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d * mult), nn.GELU(),
                                 nn.Dropout(dropout), nn.Linear(d * mult, d))

    def forward(self, x):
        return self.net(x)


class Tokenizer(nn.Module):
    """Shared per-channel multi-scale conv stem + non-overlapping patching."""

    def __init__(self, d, patch, kernels=(7, 15, 31), width=32):
        super().__init__()
        self.branches = nn.ModuleList([nn.Conv1d(1, width, k, padding=k // 2) for k in kernels])
        self.norm = nn.GroupNorm(1, width * len(kernels))
        self.proj = nn.Conv1d(width * len(kernels), d, kernel_size=patch, stride=patch)

    def forward(self, x):                      # (N, L)
        x = x.unsqueeze(1)
        h = torch.cat([F.gelu(b(x)) for b in self.branches], 1)
        return self.proj(self.norm(h)).transpose(1, 2)   # (N, T, d)


class EncoderLayer(nn.Module):
    """Criss-cross block: spatial attention (across electrodes) then temporal attention."""

    def __init__(self, d, h, dropout):
        super().__init__()
        self.n_s, self.att_s = nn.LayerNorm(d), MHA(d, h, dropout)
        self.n_t, self.att_t = nn.LayerNorm(d), MHA(d, h, dropout)
        self.ff = FFN(d, dropout=dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, eeg_mask):            # x (B, C, T, d)
        B, C, T, d = x.shape
        xs = x.permute(0, 2, 1, 3).reshape(B * T, C, d)
        xs = xs + self.drop(self.att_s(self.n_s(xs), key_mask=eeg_mask.repeat_interleave(T, 0)))
        xt = xs.view(B, T, C, d).permute(0, 2, 1, 3).reshape(B * C, T, d)
        xt = xt + self.drop(self.att_t(self.n_t(xt)))
        xt = xt + self.drop(self.ff(xt))
        return xt.view(B, C, T, d)


class GeoBias(nn.Module):
    """Lead-field-inspired additive attention bias (B, H, K, C)."""

    def __init__(self, h, hidden=64, scale_mm=80.0):
        super().__init__()
        self.scale = scale_mm
        self.mlp = nn.Sequential(nn.Linear(4, hidden), nn.GELU(), nn.Linear(hidden, h))
        nn.init.zeros_(self.mlp[-1].weight)
        nn.init.zeros_(self.mlp[-1].bias)
        self.decay = nn.Parameter(torch.linspace(0.5, 3.0, h))

    def forward(self, tgt_pos, eeg_pos):
        rel = (tgt_pos[:, :, None, :] - eeg_pos[:, None, :, :]) / self.scale
        dist = rel.norm(dim=-1, keepdim=True)
        b = self.mlp(torch.cat([rel, dist], -1)) - F.softplus(self.decay) * dist
        return b.permute(0, 3, 1, 2)


class DecoderLayer(nn.Module):
    """geo cross-attention (same t) -> contact self-attention (same t) -> temporal self-attention."""

    def __init__(self, d, h, dropout):
        super().__init__()
        self.h = h
        self.n_x, self.att_x = nn.LayerNorm(d), MHA(d, h, dropout)
        self.n_k, self.att_k = nn.LayerNorm(d), MHA(d, h, dropout)
        self.n_t, self.att_t = nn.LayerNorm(d), MHA(d, h, dropout)
        self.ff = FFN(d, dropout=dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, q, mem, geo, eeg_mask, tgt_mask):
        B, K, T, d = q.shape
        C = mem.shape[1]
        qx = q.permute(0, 2, 1, 3).reshape(B * T, K, d)
        kv = mem.permute(0, 2, 1, 3).reshape(B * T, C, d)
        bias = geo.unsqueeze(1).expand(B, T, self.h, K, C).reshape(B * T, self.h, K, C)
        qx = qx + self.drop(self.att_x(self.n_x(qx), ctx=kv, bias=bias,
                                       key_mask=eeg_mask.repeat_interleave(T, 0)))
        qx = qx + self.drop(self.att_k(self.n_k(qx), key_mask=tgt_mask.repeat_interleave(T, 0)))
        qt = qx.view(B, T, K, d).permute(0, 2, 1, 3).reshape(B * K, T, d)
        qt = qt + self.drop(self.att_t(self.n_t(qt)))
        qt = qt + self.drop(self.ff(qt))
        return qt.view(B, K, T, d)


class SubjectAdapter(nn.Module):
    """Per-contact calibration parameters (zero/identity initialised)."""

    def __init__(self, n_contacts, d):
        super().__init__()
        self.q_delta = nn.Parameter(torch.zeros(n_contacts, d))
        self.gain = nn.Parameter(torch.ones(n_contacts))
        self.bias = nn.Parameter(torch.zeros(n_contacts))


class LeadFormer(nn.Module):
    def __init__(self, d=128, heads=4, enc_layers=4, dec_layers=3, patch=16,
                 max_tokens=512, dropout=0.1):
        super().__init__()
        self.d, self.patch = d, patch
        self.tokenizer = Tokenizer(d, patch)
        self.eeg_pos = FourierXYZ(d, seed=1)
        self.tgt_pos = FourierXYZ(d, seed=2)
        self.register_buffer("time_pe", sinusoid_table(max_tokens, d), persistent=False)
        self.encoder = nn.ModuleList([EncoderLayer(d, heads, dropout) for _ in range(enc_layers)])
        self.enc_norm = nn.LayerNorm(d)
        self.geo = GeoBias(heads)
        self.query_type = nn.Parameter(torch.zeros(d))
        self.decoder = nn.ModuleList([DecoderLayer(d, heads, dropout) for _ in range(dec_layers)])
        self.dec_norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, patch)
        self.refine = nn.Sequential(nn.Conv1d(1, 16, 9, padding=4), nn.GELU(), nn.Conv1d(16, 1, 9, padding=4))
        nn.init.zeros_(self.refine[-1].weight)
        nn.init.zeros_(self.refine[-1].bias)
        self.logvar_head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Linear(d, 1))

    # ------------------------------------------------------------------ parts
    def encode(self, eeg, eeg_pos, eeg_mask):
        B, C, L = eeg.shape
        assert L % self.patch == 0, "window length must be a multiple of patch"
        T = L // self.patch
        x = self.tokenizer(eeg.reshape(B * C, L)).view(B, C, T, self.d)
        x = x + self.eeg_pos(eeg_pos)[:, :, None, :] + self.time_pe[:T][None, None]
        for layer in self.encoder:
            x = layer(x, eeg_mask)
        return self.enc_norm(x)

    def decode(self, mem, eeg_pos, eeg_mask, tgt_pos, tgt_mask, cidx=None, adapter=None):
        B, C, T, d = mem.shape
        K = tgt_pos.shape[1]
        q0 = self.tgt_pos(tgt_pos) + self.query_type
        if adapter is not None:
            q0 = q0 + adapter.q_delta[cidx]
        q = q0[:, :, None, :] + self.time_pe[:T][None, None]
        geo = self.geo(tgt_pos, eeg_pos)
        for layer in self.decoder:
            q = layer(q, mem, geo, eeg_mask, tgt_mask)
        q = self.dec_norm(q)
        y = self.head(q).reshape(B * K, 1, T * self.patch)
        y = (y + self.refine(y)).view(B, K, T * self.patch)
        logvar = self.logvar_head(q.mean(2)).squeeze(-1)
        if adapter is not None:
            y = y * adapter.gain[cidx][..., None] + adapter.bias[cidx][..., None]
        return y, logvar

    def forward(self, eeg, eeg_pos, eeg_mask, tgt_pos, tgt_mask, cidx=None, adapter=None):
        mem = self.encode(eeg, eeg_pos, eeg_mask)
        return self.decode(mem, eeg_pos, eeg_mask, tgt_pos, tgt_mask, cidx, adapter)

    def param_groups(self):
        enc = [p for n, p in self.named_parameters()
               if n.startswith(("tokenizer.", "eeg_pos.", "encoder.", "enc_norm."))]
        enc_ids = {id(p) for p in enc}
        dec = [p for p in self.parameters() if id(p) not in enc_ids]
        return enc, dec

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [ ]:
%%writefile {CODE_DIR}/s2i/losses.py
"""
Losses.

L = NLL(observability)  +  w_r * (1 - Pearson)  +  w_s * multi-resolution STFT

* NLL: per-contact heteroscedastic Gaussian on the window MSE. Contacts the model
  cannot predict get a large variance and contribute small gradients, so model
  capacity is not wasted on scalp-invisible sources.
* Multi-resolution STFT replaces CAST's single full-window |FFT| loss (which had
  no time resolution and was dominated by noise bins).
"""
import torch


def masked_mean(x, mask):
    m = mask.to(x.dtype)
    return (x * m).sum() / m.sum().clamp_min(1.0)


def pearson(pred, tgt, eps=1e-6):
    """(B, K, L) -> (B, K)."""
    p = pred - pred.mean(-1, keepdim=True)
    t = tgt - tgt.mean(-1, keepdim=True)
    return (p * t).sum(-1) / torch.sqrt((p * p).sum(-1) * (t * t).sum(-1) + eps)


def mr_stft(pred, tgt, mask, ffts=(32, 64, 128)):
    x, y = pred[mask], tgt[mask]
    if x.numel() == 0:
        return pred.new_zeros(())
    loss = 0.0
    for n in ffts:
        w = torch.hann_window(n, device=x.device)
        X = torch.stft(x, n, hop_length=n // 4, window=w, return_complex=True).abs()
        Y = torch.stft(y, n, hop_length=n // 4, window=w, return_complex=True).abs()
        sc = torch.linalg.norm(Y - X, dim=(-2, -1)) / torch.linalg.norm(Y, dim=(-2, -1)).clamp_min(1e-6)
        lm = (torch.log(X + 1e-4) - torch.log(Y + 1e-4)).abs().mean(dim=(-2, -1))
        loss = loss + (sc + lm).mean()
    return loss / len(ffts)


def total_loss(pred, logvar, tgt, mask, w_r=1.0, w_s=0.1):
    pred, logvar, tgt = pred.float(), logvar.float(), tgt.float()
    mse = ((pred - tgt) ** 2).mean(-1)
    lv = logvar.clamp(-5.0, 3.0)
    nll = 0.5 * (mse * torch.exp(-lv) + lv)
    r = pearson(pred, tgt)
    l_nll = masked_mean(nll, mask)
    l_r = masked_mean(1.0 - r, mask)
    l_s = mr_stft(pred, tgt, mask) if w_s > 0 else pred.new_zeros(())
    loss = l_nll + w_r * l_r + w_s * l_s
    return loss, {"nll": l_nll.item(), "r": 1.0 - l_r.item(), "stft": float(l_s),
                  "mse": masked_mean(mse, mask).item()}

In [ ]:
%%writefile {CODE_DIR}/s2i/metrics.py
"""
Evaluation metrics, computed per contact on the concatenation of all test trials.

Differences vs. old compute_metrics():
* R^2 is NOT clipped at 0 (clipping max(R^2, 0) hid failures and inflated means).
* Band-limited waveform r and band envelope r are reported, so we can see in
  which frequency range information actually exists.
* Filtering is done per trial (no filter ringing across trial boundaries).
"""
import numpy as np
from scipy.signal import butter, hilbert, sosfiltfilt

BANDS = {"delta": (1, 4), "theta": (4, 8), "alpha": (8, 13), "beta": (13, 30), "gamma": (30, 70)}


def _r_rows(P, T):
    P = P - P.mean(1, keepdims=True)
    T = T - T.mean(1, keepdims=True)
    den = np.sqrt((P * P).sum(1) * (T * T).sum(1)) + 1e-12
    return (P * T).sum(1) / den


def _r2_rows(P, T):
    ss_res = ((T - P) ** 2).sum(1)
    ss_tot = ((T - T.mean(1, keepdims=True)) ** 2).sum(1) + 1e-12
    return 1.0 - ss_res / ss_tot


def per_contact_metrics(preds, tgts, fs=200, bands=True):
    """preds/tgts: lists of (K, L_trial) arrays. Returns dict of (K,) arrays."""
    P = np.concatenate(preds, 1).astype(np.float64)
    T = np.concatenate(tgts, 1).astype(np.float64)
    out = {"r": _r_rows(P, T), "r2": _r2_rows(P, T),
           "rmse": np.sqrt(((P - T) ** 2).mean(1))}
    if bands:
        for name, (lo, hi) in BANDS.items():
            sos = butter(4, [lo, min(hi, fs / 2 - 1)], btype="band", fs=fs, output="sos")
            pb, tb, pe, te = [], [], [], []
            for p, t in zip(preds, tgts):
                if p.shape[1] < 3 * 2 * len(sos) * 3:
                    continue
                fp, ft = sosfiltfilt(sos, p, axis=1), sosfiltfilt(sos, t, axis=1)
                pb.append(fp)
                tb.append(ft)
                pe.append(np.abs(hilbert(fp, axis=1)))
                te.append(np.abs(hilbert(ft, axis=1)))
            if pb:
                out[f"r_{name}"] = _r_rows(np.concatenate(pb, 1), np.concatenate(tb, 1))
                out[f"env_{name}"] = _r_rows(np.concatenate(pe, 1), np.concatenate(te, 1))
    return out


def summarize(m):
    return {k: float(np.nanmean(v)) for k, v in m.items()}

In [ ]:
%%writefile {CODE_DIR}/s2i/baselines.py
"""
Lagged ridge-regression baseline (within-subject, trained on the SAME calibration
trials the deep model gets). The old paper compared against numbers reported in
other papers on other data; a claim of improvement is only meaningful against a
baseline run on the identical split.
"""
import numpy as np

DEFAULT_LAGS = (-40, -20, -10, -5, 0, 5, 10, 20, 40)   # samples @200 Hz = +-200 ms


def lagged(e, lags=DEFAULT_LAGS):
    C, L = e.shape
    out = np.zeros((C * len(lags), L), dtype=np.float32)
    for j, lag in enumerate(lags):
        blk = out[j * C:(j + 1) * C]
        if lag > 0:
            blk[:, lag:] = e[:, :-lag]
        elif lag < 0:
            blk[:, :lag] = e[:, -lag:]
        else:
            blk[:] = e
    return out


def _stack(trials_e, trials_y, lags):
    X = np.concatenate([lagged(e, lags) for e in trials_e], 1).T
    Y = np.concatenate(trials_y, 1).T
    return X.astype(np.float64), Y.astype(np.float64)


def _r_cols(P, T):
    P = P - P.mean(0)
    T = T - T.mean(0)
    return (P * T).sum(0) / (np.sqrt((P * P).sum(0) * (T * T).sum(0)) + 1e-12)


class LaggedRidge:
    def __init__(self, lags=DEFAULT_LAGS, alphas=(1e-3, 1e-2, 1e-1, 1, 10, 100, 1000)):
        self.lags, self.alphas = lags, alphas

    def _solve(self, X, Y):
        mu, sd = X.mean(0), X.std(0) + 1e-8
        Xs = (X - mu) / sd
        ym = Y.mean(0)
        G = Xs.T @ Xs
        lam, Q = np.linalg.eigh(G)
        QtXtY = Q.T @ (Xs.T @ (Y - ym))
        return mu, sd, ym, lam, Q, QtXtY

    def fit(self, trials_e, trials_y, val_frac=0.2):
        n = len(trials_e)
        n_tr = max(1, int(round(n * (1 - val_frac))))
        scale = None
        if n - n_tr >= 1:
            X, Y = _stack(trials_e[:n_tr], trials_y[:n_tr], self.lags)
            Xv, Yv = _stack(trials_e[n_tr:], trials_y[n_tr:], self.lags)
            mu, sd, ym, lam, Q, QtXtY = self._solve(X, Y)
            scale = lam.mean()
            Xvs = (Xv - mu) / sd
            scores = []
            for a in self.alphas:
                W = Q @ (QtXtY / (lam + a * scale)[:, None])
                scores.append(_r_cols(Xvs @ W + ym, Yv))
            best = np.argmax(np.stack(scores), 0)          # per-contact alpha
        X, Y = _stack(trials_e, trials_y, self.lags)
        mu, sd, ym, lam, Q, QtXtY = self._solve(X, Y)
        scale = lam.mean()
        if n - n_tr < 1:
            best = np.full(Y.shape[1], self.alphas.index(10) if 10 in self.alphas else 0)
        W = np.zeros((X.shape[1], Y.shape[1]))
        for ai, a in enumerate(self.alphas):
            cols = np.where(best == ai)[0]
            if len(cols):
                W[:, cols] = Q @ (QtXtY[:, cols] / (lam + a * scale)[:, None])
        self.mu, self.sd, self.ym, self.W = mu, sd, ym, W
        return self

    def predict(self, trials_e):
        out = []
        for e in trials_e:
            X = (lagged(e, self.lags).T - self.mu) / self.sd
            out.append((X @ self.W + self.ym).T.astype(np.float32))
        return out

In [ ]:
%%writefile {CODE_DIR}/s2i/inference.py
"""Inference helpers: full-trial reconstruction with weighted overlap-add, streaming val r."""
import numpy as np
import torch


@torch.no_grad()
def predict_trials(model, subj, trial_idx, win, stride, device, adapter=None, batch=16, amp=True):
    """Returns (preds, tgts, logvar_mean) with preds/tgts lists of (K, L) float32 arrays."""
    model.eval()
    w = torch.from_numpy((0.1 + 0.9 * np.hanning(win)).astype(np.float32)).to(device)
    K = subj.n_targets
    eeg_pos = subj.eeg_pos.to(device)[None]
    tgt_pos = subj.tgt_pos.to(device)[None]
    C = eeg_pos.shape[1]
    eeg_mask = torch.ones(1, C, dtype=torch.bool, device=device)
    tgt_mask = torch.ones(1, K, dtype=torch.bool, device=device)
    cidx = torch.arange(K, device=device)[None]
    preds, tgts, lv_sum, lv_n = [], [], torch.zeros(K, device=device), 0
    for ti in trial_idx:
        e, y = subj.trial(ti)
        e = e.float().to(device)
        L = e.shape[1]
        Lp = max(L, win)
        if Lp > L:
            e = torch.nn.functional.pad(e, (0, Lp - L))
        starts = list(range(0, Lp - win + 1, stride))
        if starts[-1] != Lp - win:
            starts.append(Lp - win)
        acc = torch.zeros(K, Lp, device=device)
        wsum = torch.zeros(Lp, device=device)
        for b0 in range(0, len(starts), batch):
            st = starts[b0:b0 + batch]
            xb = torch.stack([e[:, s:s + win] for s in st])
            n = len(st)
            with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
                pr, lv = model(xb, eeg_pos.expand(n, -1, -1), eeg_mask.expand(n, -1),
                               tgt_pos.expand(n, -1, -1), tgt_mask.expand(n, -1),
                               cidx.expand(n, -1), adapter)
            pr, lv = pr.float(), lv.float()
            for j, s in enumerate(st):
                acc[:, s:s + win] += pr[j] * w
                wsum[s:s + win] += w
            lv_sum += lv.sum(0)
            lv_n += n
        preds.append((acc / wsum)[:, :L].cpu().numpy())
        tgts.append(y.float().numpy())
    return preds, tgts, (lv_sum / max(lv_n, 1)).cpu().numpy()


@torch.no_grad()
def streaming_r(model, loader, n_subjects, n_targets, device, amp=True, adapter=None):
    """Mean (over subjects) of mean (over contacts) Pearson r, accumulated over windows."""
    model.eval()
    stats = [torch.zeros(k, 5, dtype=torch.float64, device=device) for k in n_targets]
    counts = [0] * n_subjects
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        with torch.autocast(device_type=device.type, enabled=amp and device.type == "cuda"):
            pr, _ = model(b["eeg"], b["eeg_pos"], b["eeg_mask"], b["tgt_pos"], b["tgt_mask"],
                          b["cidx"], adapter)
        pr = pr.double()
        tg = b["tgt"].double()
        L = pr.shape[-1]
        for i in range(pr.shape[0]):
            s = int(b["subj"][i])
            k = int(b["tgt_mask"][i].sum())
            p, t, ci = pr[i, :k], tg[i, :k], b["cidx"][i, :k]
            v = torch.stack([p.sum(-1), t.sum(-1), (p * p).sum(-1), (t * t).sum(-1), (p * t).sum(-1)], -1)
            stats[s].index_add_(0, ci, v)
            counts[s] += L
    per_subj = []
    for s, st in enumerate(stats):
        n = counts[s]
        if n == 0:
            continue
        sp, sq, spp, sqq, spq = st.unbind(-1)
        num = n * spq - sp * sq
        den = torch.sqrt((n * spp - sp ** 2).clamp_min(1e-12) * (n * sqq - sq ** 2).clamp_min(1e-12))
        per_subj.append((num / den).mean().item())
    return float(np.mean(per_subj)) if per_subj else float("nan")

In [ ]:
%%writefile {CODE_DIR}/s2i/train_loso.py
"""
Leave-one-subject-out protocol for LeadFormer.

Per fold (test subject S):
  1. Joint pre-training on all other subjects (mixed, subject-balanced batches;
     old code trained subjects *sequentially* and threw the decoder away).
     Model selection = zero-shot r on held-out VALIDATION subjects
     (old code picked the epoch with the lowest *training* loss).
  2. Zero-shot evaluation on S (no data from S used at all).
  3. Calibration on the first `calib_frac` of S's trials (chronological), with an
     internal chronological validation split for early stopping; adapters +
     decoder are tuned, encoder frozen.
  4. Lagged-ridge baseline trained on exactly the same calibration trials.
  5. Everything evaluated on the remaining trials (after a 1-trial gap).

Primary metric: mean over ALL contacts of S, then mean over subjects. No
outcome-based subject/channel subsets.

Example
-------
python -m s2i.train_loso --data c:\\iEEG_EEG\\data\\s2i_v1 --out runs\\bip_car --target bip --eeg_ref car
"""
import argparse
import copy
import json
import logging
import os
import random
import sys
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from s2i.baselines import LaggedRidge
from s2i.data.dataset import Subject, WindowDataset, chrono_split, collate
from s2i.inference import predict_trials, streaming_r
from s2i.losses import total_loss
from s2i.metrics import per_contact_metrics, summarize
from s2i.model.leadformer import LeadFormer, SubjectAdapter


# ----------------------------------------------------------------------------- utils
def fix_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def augment(b, p_drop=0.4, min_ch=4, noise=0.05):
    """Scalp-channel dropout (montage robustness), per-channel gain, additive noise."""
    eeg, mask = b["eeg"], b["eeg_mask"]
    B, C, _ = eeg.shape
    rate = torch.rand(B, 1, device=eeg.device) * p_drop
    drop = (torch.rand(B, C, device=eeg.device) < rate) & mask
    keep_n = (mask & ~drop).sum(1)
    drop[keep_n < min_ch] = False
    new_mask = mask & ~drop
    gain = 0.8 + 0.4 * torch.rand(B, C, 1, device=eeg.device)
    eeg = (eeg * gain + noise * torch.randn_like(eeg)) * new_mask[..., None]
    b = dict(b)
    b["eeg"], b["eeg_mask"] = eeg, new_mask
    return b


def to_dev(b, dev):
    return {k: v.to(dev, non_blocking=True) for k, v in b.items()}


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def make_logger(path):
    log = logging.getLogger(path)
    log.setLevel(logging.INFO)
    log.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S")
    for h in (logging.FileHandler(path, encoding="utf-8"), logging.StreamHandler(sys.stdout)):
        h.setFormatter(fmt)
        log.addHandler(h)
    return log


def build_model(a):
    return LeadFormer(d=a.d, heads=a.heads, enc_layers=a.enc_layers, dec_layers=a.dec_layers,
                      patch=a.patch, dropout=a.dropout)


# ----------------------------------------------------------------------------- stage 1
def pretrain(a, train_subj, val_subj, dev, log, ckpt_path=None):
    model = build_model(a).to(dev)
    log.info(f"  params: {model.count_parameters() / 1e6:.2f} M")
    tr_ds = WindowDataset(train_subj, {s.sid: list(range(s.n_trials)) for s in train_subj},
                          a.win, "train", k_max=a.k_max, crops_per_trial=1, seed=a.seed)
    sampler = WeightedRandomSampler(tr_ds.subject_balanced_weights(), a.steps * a.bs, replacement=True)
    tr_ld = DataLoader(tr_ds, a.bs, sampler=sampler, collate_fn=collate, num_workers=0, drop_last=True)
    va_trials = {s.sid: list(range(min(s.n_trials, a.val_trials))) for s in val_subj}
    va_ds = WindowDataset(val_subj, va_trials, a.win, "eval", stride=a.win)
    va_ld = DataLoader(va_ds, a.bs, shuffle=False, collate_fn=collate, num_workers=0)

    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
    total = a.epochs * a.steps
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=total, pct_start=0.05,
                                                anneal_strategy="cos")
    scaler = make_scaler(a.amp and dev.type == "cuda")
    best, best_state, bad, start = -1e9, None, 0, 0
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        best, best_state, bad, start = ck["best"], ck["best_state"], ck["bad"], ck["epoch"] + 1
        log.info(f"  resumed from epoch {start} (best val r={best:.4f})")
    for ep in range(start, a.epochs):
        if bad >= a.patience:
            break
        model.train()
        t0, agg = time.time(), {"nll": 0.0, "r": 0.0, "stft": 0.0}
        for b in tr_ld:
            b = augment(to_dev(b, dev))
            with torch.autocast(device_type=dev.type, enabled=a.amp and dev.type == "cuda"):
                pred, lv = model(b["eeg"], b["eeg_pos"], b["eeg_mask"], b["tgt_pos"], b["tgt_mask"])
            loss, parts = total_loss(pred, lv, b["tgt"], b["tgt_mask"], a.w_r, a.w_s)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            for k in agg:
                agg[k] += parts[k] / a.steps
        val_r = streaming_r(model, va_ld, len(val_subj), [s.n_targets for s in val_subj], dev, a.amp)
        log.info(f"  ep {ep + 1:3d} | train nll={agg['nll']:.3f} r={agg['r']:.3f} stft={agg['stft']:.3f}"
                 f" | val zero-shot r={val_r:.4f} | {time.time() - t0:.0f}s")
        if val_r > best:
            best, bad = val_r, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if ckpt_path:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "best": best, "best_state": best_state,
                        "bad": bad, "epoch": ep}, ckpt_path + ".tmp")
            os.replace(ckpt_path + ".tmp", ckpt_path)
        if bad >= a.patience:
            log.info("  early stop")
            break
    model.load_state_dict(best_state)
    return model, best


# ----------------------------------------------------------------------------- stage 3
def calibrate(a, base, subj, cal_trials, dev, log):
    model = copy.deepcopy(base).to(dev)
    adapter = SubjectAdapter(subj.n_targets, a.d).to(dev)
    enc, dec = model.param_groups()
    for p in enc:
        p.requires_grad_(False)
    fit_tr, fit_va = chrono_split(len(cal_trials), 0.8, gap=0)
    fit_tr = [cal_trials[i] for i in fit_tr]
    fit_va = [cal_trials[i] for i in fit_va] or fit_tr
    tr_ds = WindowDataset([subj], {subj.sid: fit_tr}, a.win, "train", k_max=a.k_max,
                          crops_per_trial=4, seed=a.seed)
    va_ds = WindowDataset([subj], {subj.sid: fit_va}, a.win, "eval", stride=a.win // 2)
    va_ld = DataLoader(va_ds, a.bs, shuffle=False, collate_fn=collate)
    opt = torch.optim.AdamW([{"params": dec, "lr": a.calib_lr_dec, "weight_decay": a.wd},
                             {"params": adapter.parameters(), "lr": a.calib_lr_adapter, "weight_decay": 0.0}])
    scaler = make_scaler(a.amp and dev.type == "cuda")
    best = streaming_r(model, va_ld, 1, [subj.n_targets], dev, a.amp, adapter)
    log.info(f"  calib start (zero-shot on calib-val) r={best:.4f}")
    best_state = (copy.deepcopy(model.state_dict()), copy.deepcopy(adapter.state_dict()))
    bad, ep = 0, -1
    for ep in range(a.calib_epochs):
        model.train()
        ld = DataLoader(tr_ds, a.bs, shuffle=True, collate_fn=collate, drop_last=len(tr_ds) > a.bs)
        for b in ld:
            b = augment(to_dev(b, dev))
            with torch.autocast(device_type=dev.type, enabled=a.amp and dev.type == "cuda"):
                pred, lv = model(b["eeg"], b["eeg_pos"], b["eeg_mask"], b["tgt_pos"], b["tgt_mask"],
                                 b["cidx"], adapter)
            loss, _ = total_loss(pred, lv, b["tgt"], b["tgt_mask"], a.w_r, a.w_s)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(list(dec) + list(adapter.parameters()), 1.0)
            scaler.step(opt)
            scaler.update()
        r = streaming_r(model, va_ld, 1, [subj.n_targets], dev, a.amp, adapter)
        if r > best:
            best, bad = r, 0
            best_state = (copy.deepcopy(model.state_dict()), copy.deepcopy(adapter.state_dict()))
        else:
            bad += 1
            if bad >= a.calib_patience:
                break
    model.load_state_dict(best_state[0])
    adapter.load_state_dict(best_state[1])
    log.info(f"  calib done: best calib-val r={best:.4f} after {ep + 1} epochs")
    return model, adapter


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--target", choices=["bip", "mono"], default="bip")
    ap.add_argument("--eeg_ref", choices=["car", "orig"], default="car")
    ap.add_argument("--subjects", nargs="*", default=None, help="subset of subjects to load")
    ap.add_argument("--folds", nargs="*", default=None, help="test subjects to run (default: all)")
    ap.add_argument("--min_trials", type=int, default=20)
    # model
    ap.add_argument("--d", type=int, default=128)
    ap.add_argument("--heads", type=int, default=4)
    ap.add_argument("--enc_layers", type=int, default=4)
    ap.add_argument("--dec_layers", type=int, default=3)
    ap.add_argument("--patch", type=int, default=16)
    ap.add_argument("--dropout", type=float, default=0.1)
    # training
    ap.add_argument("--win", type=int, default=800)
    ap.add_argument("--k_max", type=int, default=48)
    ap.add_argument("--bs", type=int, default=16)
    ap.add_argument("--lr", type=float, default=5e-4)
    ap.add_argument("--wd", type=float, default=0.05)
    ap.add_argument("--epochs", type=int, default=60)
    ap.add_argument("--steps", type=int, default=300, help="optimizer steps per epoch")
    ap.add_argument("--patience", type=int, default=10)
    ap.add_argument("--n_val_subj", type=int, default=2)
    ap.add_argument("--val_trials", type=int, default=60)
    ap.add_argument("--w_r", type=float, default=1.0)
    ap.add_argument("--w_s", type=float, default=0.1)
    # calibration
    ap.add_argument("--calib_frac", type=float, default=0.2)
    ap.add_argument("--calib_epochs", type=int, default=40)
    ap.add_argument("--calib_patience", type=int, default=6)
    ap.add_argument("--calib_lr_dec", type=float, default=1e-4)
    ap.add_argument("--calib_lr_adapter", type=float, default=3e-3)
    ap.add_argument("--no_calib", action="store_true")
    ap.add_argument("--no_ridge", action="store_true")
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    a.amp = not a.no_amp
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log = make_logger(os.path.join(a.out, "run.log"))
    dev = torch.device(a.device)
    log.info(f"config: {vars(a)}")
    with open(os.path.join(a.out, "config.json"), "w") as f:
        json.dump(vars(a), f, indent=2)

    files = sorted(f for f in os.listdir(a.data) if f.endswith(".pt"))
    subjects = []
    for f in files:
        sid = f[:-3]
        if a.subjects and sid not in a.subjects:
            continue
        s = Subject(os.path.join(a.data, f), a.target, a.eeg_ref)
        if s.n_trials < a.min_trials or s.n_targets == 0:
            log.info(f"skip {sid}: {s.n_trials} trials, {s.n_targets} targets")
            continue
        subjects.append(s)
        log.info(f"{sid}: {s.n_trials} trials, {len(s.eeg_names)} scalp, {s.n_targets} targets, "
                 f"diag={s.diagnostics}")
    folds = [s for s in subjects if not a.folds or s.sid in a.folds]

    all_res = []
    for fi, test in enumerate(folds):
        fold_path = os.path.join(a.out, f"fold_{test.sid}.json")
        if os.path.exists(fold_path):
            log.info(f"[{test.sid}] already done, loading")
            with open(fold_path) as f:
                all_res.append(json.load(f))
            continue
        fix_seed(a.seed + fi)
        log.info("=" * 78)
        log.info(f"[{fi + 1}/{len(folds)}] TEST {test.sid}")
        pool = [s for s in subjects if s.sid != test.sid]
        rng = np.random.default_rng(a.seed + fi)
        order = rng.permutation(len(pool))
        val_subj = [pool[i] for i in order[:a.n_val_subj]]
        train_subj = [pool[i] for i in order[a.n_val_subj:]]
        log.info(f"  train={[s.sid for s in train_subj]} val={[s.sid for s in val_subj]}")

        t0 = time.time()
        model_path = os.path.join(a.out, f"model_{test.sid}.pt")
        ckpt_path = os.path.join(a.out, f"ckpt_{test.sid}.pt")
        if os.path.exists(model_path):
            model = build_model(a).to(dev)
            saved = torch.load(model_path, map_location=dev, weights_only=False)
            model.load_state_dict(saved["state"])
            best_val = saved["best_val"]
            log.info(f"  loaded pretrained model (val r={best_val:.4f})")
        else:
            model, best_val = pretrain(a, train_subj, val_subj, dev, log, ckpt_path)
            torch.save({"state": model.state_dict(), "best_val": best_val}, model_path)
            if os.path.exists(ckpt_path):
                os.remove(ckpt_path)

        cal_tr, test_tr = chrono_split(test.n_trials, a.calib_frac, gap=1)
        log.info(f"  split: calib {len(cal_tr)} trials | test {len(test_tr)} trials")
        res = {"subject": test.sid, "val_subjects": [s.sid for s in val_subj], "best_val_r": best_val,
               "names": test.tgt_names, "region": test.tgt_region, "depth": test.tgt_depth,
               "n_calib_trials": len(cal_tr), "n_test_trials": len(test_tr),
               "calib_minutes": sum(test.trial_len(i) for i in cal_tr) / 200 / 60,
               "diagnostics": test.diagnostics}

        # zero-shot
        P, T, lv_test = predict_trials(model, test, test_tr, a.win, a.win // 2, dev, amp=a.amp)
        m = per_contact_metrics(P, T)
        res["zero_shot"] = {k: v.tolist() for k, v in m.items()}
        # observability predicted from calibration-trial INPUTS only (no targets used)
        _, _, lv_cal = predict_trials(model, test, cal_tr, a.win, a.win // 2, dev, amp=a.amp)
        res["pred_logvar_calib"] = lv_cal.tolist()
        log.info(f"  ZERO-SHOT  : {fmt(summarize(m))}")

        if not a.no_calib:
            cm, ad = calibrate(a, model, test, cal_tr, dev, log)
            P, T, _ = predict_trials(cm, test, test_tr, a.win, a.win // 2, dev, ad, amp=a.amp)
            m = per_contact_metrics(P, T)
            res["calibrated"] = {k: v.tolist() for k, v in m.items()}
            log.info(f"  CALIBRATED : {fmt(summarize(m))}")
            torch.save({"state": cm.state_dict(), "adapter": ad.state_dict(),
                        "n_targets": test.n_targets, "cal_trials": cal_tr, "test_trials": test_tr},
                       os.path.join(a.out, f"calib_{test.sid}.pt"))
            del cm, ad

        if not a.no_ridge:
            ce = [test.trial(i)[0].float().numpy() for i in cal_tr]
            cy = [test.trial(i)[1].float().numpy() for i in cal_tr]
            rg = LaggedRidge().fit(ce, cy)
            te = [test.trial(i)[0].float().numpy() for i in test_tr]
            ty = [test.trial(i)[1].float().numpy() for i in test_tr]
            m = per_contact_metrics(rg.predict(te), ty)
            res["ridge"] = {k: v.tolist() for k, v in m.items()}
            log.info(f"  RIDGE      : {fmt(summarize(m))}")

        res["minutes"] = (time.time() - t0) / 60
        with open(fold_path, "w") as f:
            json.dump(res, f)
        all_res.append(res)
        del model
        torch.cuda.empty_cache()

    report(all_res, a.out, log)


def fmt(s):
    keys = ["r", "r2", "r_delta", "r_theta", "r_alpha", "r_beta", "r_gamma", "env_gamma"]
    return " ".join(f"{k}={s[k]:.3f}" for k in keys if k in s)


def report(all_res, out, log):
    from scipy.stats import spearmanr, wilcoxon
    methods = [m for m in ("zero_shot", "calibrated", "ridge") if all(m in r for r in all_res)]
    lines = ["| Subject | " + " | ".join(f"{m} r" for m in methods) + " | " +
             " | ".join(f"{m} R²" for m in methods) + " |",
             "|---" * (1 + 2 * len(methods)) + "|"]
    per_subj = {m: [] for m in methods}
    for r in all_res:
        row = [r["subject"]]
        for m in methods:
            per_subj[m].append(np.nanmean(r[m]["r"]))
            row.append(f"{np.nanmean(r[m]['r']):.3f}")
        for m in methods:
            row.append(f"{np.nanmean(r[m]['r2']):.3f}")
        lines.append("| " + " | ".join(row) + " |")
    lines.append("| **MEAN** | " + " | ".join(f"**{np.mean(per_subj[m]):.3f}**" for m in methods) +
                 " | " + " | ".join(f"{np.mean([np.nanmean(r[m]['r2']) for r in all_res]):.3f}"
                                    for m in methods) + " |")
    summary = {"n_subjects": len(all_res),
               "mean_r": {m: float(np.mean(per_subj[m])) for m in methods},
               "std_r": {m: float(np.std(per_subj[m])) for m in methods}}
    if "calibrated" in methods and "ridge" in methods and len(all_res) >= 5:
        st = wilcoxon(per_subj["calibrated"], per_subj["ridge"])
        summary["wilcoxon_calibrated_vs_ridge_p"] = float(st.pvalue)
    # depth breakdown (pooled contacts)
    depth = {}
    for m in methods:
        pools = {}
        for r in all_res:
            for dpt, val in zip(r["depth"], r[m]["r"]):
                pools.setdefault(dpt, []).append(val)
        depth[m] = {k: {"n": len(v), "mean_r": float(np.nanmean(v))} for k, v in pools.items()}
    summary["depth"] = depth
    # does predicted observability (from inputs only) rank contacts correctly?
    key = "calibrated" if "calibrated" in methods else "zero_shot"
    rhos = [spearmanr(-np.asarray(r["pred_logvar_calib"]), r[key]["r"]).correlation for r in all_res]
    summary["observability_spearman_mean"] = float(np.nanmean(rhos))
    top = []
    for r in all_res:
        lv = np.asarray(r["pred_logvar_calib"])
        k = max(1, len(lv) // 4)
        top.append(np.nanmean(np.asarray(r[key]["r"])[np.argsort(lv)[:k]]))
    summary["top25pct_predicted_observable_mean_r"] = float(np.mean(top))
    with open(os.path.join(out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    with open(os.path.join(out, "summary.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n\n```json\n" + json.dumps(summary, indent=2) + "\n```\n")
    log.info("\n" + "\n".join(lines))
    log.info(json.dumps(summary, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/diagnose_reference.py
"""
Common-reference contamination test  (run this BEFORE trusting any number).

Hypothesis: in the old monopolar setup, scalp EEG and every iEEG contact were
recorded against a shared reference R:
    EEG_c  = e_c - R           iEEG_k = s_k - R
so -R is present in every input AND every target. Predicting R is trivial and
says nothing about intracranial sources. Signatures in the old logs:
  * bipolar Hipp/Amyg, WITHIN-subject (easiest setting)   -> mean r = 0.14
  * monopolar, CROSS-subject (harder setting)             -> sub-13 r = 0.78
  * "observable" sets contain 45-67 of 64-68 contacts incl. hippocampus,
    and the same patients are "good" in both GIN and ds004752 (06, 09, 13, 15)
    -> subject (recording-setup) dependent, not anatomy dependent.

This script fits the same lagged ridge on 4 input/target combinations:
                    target=mono     target=bipolar
  input=orig ref        A                 B
  input=CAR             C                 D
If A >> B and A >> C, the monopolar "reconstruction" is mostly the shared
reference. D is the honest number for intracranial reconstruction.

python -m s2i.diagnose_reference --data c:\\iEEG_EEG\\data\\s2i_v1
"""
import argparse
import os
import sys

import numpy as np
import torch

from s2i.baselines import LaggedRidge
from s2i.data.dataset import chrono_split
from s2i.metrics import per_contact_metrics


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--calib_frac", type=float, default=0.5)
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    rows = []
    for f in sorted(x for x in os.listdir(a.data) if x.endswith(".pt")):
        d = torch.load(os.path.join(a.data, f), weights_only=False)
        b = d["trial_bounds"].numpy()
        if len(b) < 10:
            continue
        tr, te = chrono_split(len(b), a.calib_frac, gap=1)

        def seg(key, idx):
            return [d[key][:, s0:s1].float().numpy() for s0, s1 in b[idx]]

        res = {}
        for ein in ("eeg_orig", "eeg_car"):
            for tgt in ("ieeg_mono", "ieeg_bip"):
                if d[tgt].shape[0] == 0:
                    res[(ein, tgt)] = np.nan
                    continue
                rg = LaggedRidge().fit(seg(ein, tr), seg(tgt, tr))
                m = per_contact_metrics(rg.predict(seg(ein, te)), seg(tgt, te), bands=False)
                res[(ein, tgt)] = float(np.mean(m["r"]))
        dg = d.get("diagnostics", {})
        rows.append((d["subject"], res, dg))
        print(f"{d['subject']}: A(orig->mono)={res[('eeg_orig', 'ieeg_mono')]:.3f} "
              f"B(orig->bip)={res[('eeg_orig', 'ieeg_bip')]:.3f} "
              f"C(car->mono)={res[('eeg_car', 'ieeg_mono')]:.3f} "
              f"D(car->bip)={res[('eeg_car', 'ieeg_bip')]:.3f} | "
              f"mono PC1={dg.get('mono_pc1_ratio', np.nan):.2f} "
              f"|corr(mean iEEG, EEG)|={dg.get('mean_abs_corr_iEEGmean_vs_EEG', np.nan):.2f}")

    def col(key):
        return np.nanmean([r[1][key] for r in rows])

    A, B = col(("eeg_orig", "ieeg_mono")), col(("eeg_orig", "ieeg_bip"))
    C, D = col(("eeg_car", "ieeg_mono")), col(("eeg_car", "ieeg_bip"))
    print("\nMEAN  A={:.3f}  B={:.3f}  C={:.3f}  D={:.3f}".format(A, B, C, D))
    if A > 1.5 * max(B, 1e-3):
        print("=> Strong evidence of shared-reference contamination in monopolar targets.\n"
              "   Report bipolar (or locally re-referenced) results as the primary outcome.")
    else:
        print("=> No strong contamination signature; monopolar results are less suspect.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/smoke_test.py
"""
End-to-end smoke test on SYNTHETIC data (no real recordings, CPU is fine).

It simulates a toy volume-conduction world that deliberately contains a shared
recording reference R in both scalp EEG and monopolar iEEG, writes subjects in
the exact format produced by preprocess_ds004752.py, then runs:
  1. diagnose_reference  (should flag A >> D, i.e. reference contamination)
  2. train_loso          (tiny model, 2 folds, a few steps) -> checks the whole pipeline

python -m s2i.smoke_test
"""
import os
import shutil
import sys
import tempfile

import numpy as np
import torch

from s2i.data.preprocess_ds004752 import common_mode_stats, make_bipolar, robust_norm

FS = 200


def _sphere_points(n, r, rng, upper=True):
    v = rng.normal(size=(n, 3))
    if upper:
        v[:, 2] = np.abs(v[:, 2])
    return r * v / np.linalg.norm(v, axis=1, keepdims=True)


def _gain(a, b, p=2.0, eps=5.0):
    d = np.linalg.norm(a[:, None] - b[None], axis=-1)
    return 1.0 / (d + eps) ** p


def make_subject(sid, rng, n_trials=30, L=8 * FS):
    n_src = 40
    src_pos = _sphere_points(n_src, 1.0, rng, upper=False) * rng.uniform(20, 70, (n_src, 1))
    all_scalp = _sphere_points(21, 90.0, rng)
    n_eeg = int(rng.choice([6, 8, 17, 19, 21]))
    eeg_pos = all_scalp[:n_eeg]
    # 3 shafts x 8 contacts going from the surface inwards
    contacts, names = [], []
    for s in range(3):
        entry = _sphere_points(1, 75.0, rng)[0]
        direction = -entry / np.linalg.norm(entry)
        for c in range(8):
            contacts.append(entry + direction * 3.5 * c)
            names.append(f"S{s}C{c + 1}")
    contacts = np.asarray(contacts)
    T = n_trials * L
    t = np.arange(T) / FS
    freqs = rng.uniform(3, 30, n_src)
    src = np.sin(2 * np.pi * freqs[:, None] * t + rng.uniform(0, 6.28, (n_src, 1)))
    src *= 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(0.1, 0.5, (n_src, 1)) * t)
    src += 0.5 * rng.normal(size=src.shape)
    R = 3.0 * np.convolve(rng.normal(size=T), np.ones(20) / 20, mode="same")   # shared reference
    G_e = _gain(eeg_pos, src_pos)
    G_i = _gain(contacts, src_pos, p=3.0)
    e_v = (G_e / G_e.sum(1, keepdims=True)) @ src - R + 0.3 * rng.normal(size=(n_eeg, T))
    i_v = (G_i / G_i.sum(1, keepdims=True)) @ src - R + 0.3 * rng.normal(size=(len(names), T))
    pairs = make_bipolar(names)
    bip_v = np.stack([i_v[a] - i_v[b] for a, b in pairs])
    diag = common_mode_stats(i_v, e_v)
    depth = ["Cortical" if c % 8 < 3 else ("Mid" if c % 8 < 5 else "Deep") for c in range(len(names))]
    d = {
        "subject": sid, "fs": FS, "eeg_names": [f"E{i}" for i in range(n_eeg)],
        "eeg_pos": torch.tensor(eeg_pos, dtype=torch.float32),
        "contact_names": names, "contact_mni": torch.tensor(contacts, dtype=torch.float32),
        "contact_region": depth, "contact_cat": depth, "contact_depth": depth,
        "bip_pairs": pairs, "bip_names": [f"{names[a]}-{names[b]}" for a, b in pairs],
        "bip_mni": torch.tensor(np.stack([(contacts[a] + contacts[b]) / 2 for a, b in pairs]),
                                dtype=torch.float32),
        "bip_region": [depth[a] for a, _ in pairs], "bip_cat": [depth[a] for a, _ in pairs],
        "bip_depth": [depth[a] for a, _ in pairs],
        "trial_bounds": torch.tensor([(i * L, (i + 1) * L) for i in range(n_trials)]),
        "trial_session": ["ses-01"] * n_trials, "trial_id": list(range(n_trials)),
        "diagnostics": diag, "n_rejected_trials": 0,
        "eeg_orig": torch.from_numpy(robust_norm(e_v).astype(np.float16)),
        "eeg_car": torch.from_numpy(robust_norm(e_v - e_v.mean(0, keepdims=True)).astype(np.float16)),
        "ieeg_mono": torch.from_numpy(robust_norm(i_v).astype(np.float16)),
        "ieeg_bip": torch.from_numpy(robust_norm(bip_v).astype(np.float16)),
    }
    return d


def main():
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    rng = np.random.default_rng(0)
    root = tempfile.mkdtemp(prefix="s2i_smoke_")
    data = os.path.join(root, "data")
    os.makedirs(data)
    for i in range(5):
        sid = f"sub-{i + 1:02d}"
        torch.save(make_subject(sid, rng), os.path.join(data, f"{sid}.pt"))
    print(f"synthetic data -> {data}")

    from s2i import diagnose_reference, train_loso
    print("\n--- diagnose_reference ---")
    sys.argv = ["diag", "--data", data]
    diagnose_reference.main()

    print("\n--- train_loso (tiny) ---")
    sys.argv = ["train", "--data", data, "--out", os.path.join(root, "run"), "--target", "bip",
                "--device", "cpu", "--no_amp", "--d", "32", "--enc_layers", "1", "--dec_layers", "1",
                "--epochs", "2", "--steps", "3", "--bs", "4", "--win", "400", "--k_max", "12",
                "--val_trials", "4", "--n_val_subj", "1", "--calib_epochs", "2",
                "--folds", "sub-01", "sub-02", "--min_trials", "10"]
    train_loso.main()
    print(f"\nSMOKE TEST PASSED. Outputs in {root}")
    if "--keep" not in sys.argv:
        shutil.rmtree(root, ignore_errors=True)


if __name__ == "__main__":
    main()

In [ ]:
#@title 5b. Kiểm tra import
import importlib, s2i
for m in ['s2i.montage', 's2i.data.dataset', 's2i.model.leadformer', 's2i.losses',
          's2i.metrics', 's2i.baselines', 's2i.inference', 's2i.train_loso']:
    importlib.import_module(m)
from s2i.model.leadformer import LeadFormer
print('OK | LeadFormer params:', round(LeadFormer().count_parameters()/1e6, 2), 'M')

## 6. Smoke test (dữ liệu giả)
Mô phỏng một "não đồ chơi" có **tham chiếu chung cố ý**, rồi chạy toàn bộ pipeline với model rất nhỏ trên CPU.
Kỳ vọng: bước diagnose báo `A >> D` (phát hiện được nhiễm tham chiếu), và in ra `SMOKE TEST PASSED`.

In [ ]:
#@title 6. Smoke test
!python -m s2i.smoke_test

In [ ]:
#@title 7. Tiền xử lý ds004752 -> Drive (bỏ qua subject đã xong)
!python -m s2i.data.preprocess_ds004752 --bids {RAW_DIR} --out {DATA_DIR}

In [ ]:
#@title 8. Tổng quan dữ liệu đã tiền xử lý
import json, pandas as pd
summ = json.load(open(f'{DATA_DIR}/summary.json'))
df = pd.DataFrame([{**{k: v for k, v in s.items() if k not in ('eeg', 'diagnostics')},
                    'n_scalp': len(s['eeg']), **s['diagnostics']} for s in summ])
pd.set_option('display.width', 200)
display(df.round(3))
print('Tổng phút:', df['minutes'].sum(), '| tổng contact mono:', df['contacts'].sum(),
      '| tổng kênh bipolar:', df['bipolar'].sum())

## 9. Kiểm định tham chiếu chung (quan trọng nhất)
Cùng một ridge regression được fit trên 4 tổ hợp:

| | target = mono | target = bipolar |
|---|---|---|
| **EEG ref gốc** | A | B |
| **EEG CAR** | C | D |

Nếu **A ≫ B** thì phần lớn những gì "tái tạo" được ở monopolar là tín hiệu của điện cực tham chiếu chung, tức là kết quả CAST cũ bị thổi phồng. **D** là con số trung thực.

In [ ]:
#@title 9. Diagnose reference contamination
!python -m s2i.diagnose_reference --data {DATA_DIR} | tee {RUNS_DIR}/diagnose_reference.txt

## 10. LOSO chính: target **bipolar**, EEG **CAR**
Với mỗi subject test: pretrain chung trên các subject còn lại (chọn model theo **2 subject validation**), đánh giá **zero-shot**, sau đó **calibrate** trên 20% trial đầu tiên, và **ridge baseline** trên cùng split.

- `MODE='fast'`: 30 epoch × 200 step, để thử nhanh.
- `MODE='full'`: 60 epoch × 300 step, early stopping.

Bị ngắt kết nối thì chạy lại cell 1 và 3, rồi chạy lại cell này: fold đã xong được bỏ qua, fold đang dở resume từ epoch cuối.

In [ ]:
#@title 10. LOSO chính (bipolar + CAR)
RUN_MAIN = f'{RUNS_DIR}/bip_car'
EXTRA = '--epochs 30 --steps 200 --patience 6' if MODE == 'fast' else ''
!python -m s2i.train_loso --data {DATA_DIR} --out {RUN_MAIN} --target bip --eeg_ref car {EXTRA}

In [ ]:
#@title 11. (Tuỳ chọn) LOSO đối chứng: monopolar + ref gốc, cùng setup với CAST cũ
RUN_CTRL = f'{RUNS_DIR}/mono_orig'
EXTRA = '--epochs 30 --steps 200 --patience 6' if MODE == 'fast' else ''
!python -m s2i.train_loso --data {DATA_DIR} --out {RUN_CTRL} --target mono --eeg_ref orig {EXTRA}

In [ ]:
#@title 12. Bảng kết quả theo subject
import json, glob, numpy as np, pandas as pd

def load_run(run):
    return [json.load(open(f)) for f in sorted(glob.glob(f'{run}/fold_*.json'))]

def subj_table(folds):
    rows = []
    for f in folds:
        row = {'subject': f['subject'], 'n_targets': len(f['names']),
               'calib_min': round(f['calib_minutes'], 1), 'val_r': f['best_val_r']}
        for m in ('zero_shot', 'calibrated', 'ridge'):
            if m in f:
                row[f'{m}_r'] = np.nanmean(f[m]['r'])
                row[f'{m}_R2'] = np.nanmean(f[m]['r2'])
        rows.append(row)
    return pd.DataFrame(rows)

RESULTS = {}
for name in ['bip_car', 'mono_orig']:
    run = f'{RUNS_DIR}/{name}'
    folds = load_run(run)
    if not folds:
        continue
    RESULTS[name] = folds
    df = subj_table(folds)
    print(f'\n========== {name}  ({len(folds)} subjects) ==========')
    display(df.round(3))
    display(df.drop(columns=['subject']).agg(['mean', 'std']).round(3))
    if os.path.exists(f'{run}/summary.json'):
        print(json.dumps(json.load(open(f'{run}/summary.json')), indent=1)[:3000])

In [ ]:
#@title 13. Biểu đồ: so sánh phương pháp, độ sâu, băng tần, độ quan sát được
import matplotlib.pyplot as plt
FIG_DIR = f'{RUNS_DIR}/figures'; os.makedirs(FIG_DIR, exist_ok=True)
methods = ['zero_shot', 'calibrated', 'ridge']
colors = {'zero_shot': '#8e9aaf', 'calibrated': '#e76f51', 'ridge': '#2a9d8f'}

for name, folds in RESULTS.items():
    ms = [m for m in methods if all(m in f for f in folds)]
    fig, ax = plt.subplots(2, 2, figsize=(15, 9))
    # (a) theo subject
    subs = [f['subject'] for f in folds]; x = np.arange(len(subs)); w = 0.8 / len(ms)
    for i, m in enumerate(ms):
        ax[0, 0].bar(x + i * w, [np.nanmean(f[m]['r']) for f in folds], w, label=m, color=colors[m])
    ax[0, 0].set_xticks(x + w * (len(ms) - 1) / 2); ax[0, 0].set_xticklabels(subs, rotation=45)
    ax[0, 0].axhline(0, c='k', lw=.5); ax[0, 0].set_ylabel('mean r (all contacts)')
    ax[0, 0].set_title(f'{name}: per subject'); ax[0, 0].legend()
    # (b) theo độ sâu (pooled contacts)
    key = 'calibrated' if 'calibrated' in ms else 'zero_shot'
    pools = {}
    for f in folds:
        for d, v in zip(f['depth'], f[key]['r']):
            pools.setdefault(d, []).append(v)
    order = [d for d in ['Cortical', 'Mid', 'Deep', 'Mixed', 'Unknown'] if d in pools]
    ax[0, 1].boxplot([pools[d] for d in order], labels=[f'{d}\nN={len(pools[d])}' for d in order], showfliers=False)
    ax[0, 1].axhline(0, c='k', lw=.5); ax[0, 1].set_title(f'{key} r by depth')
    # (c) băng tần: waveform r và envelope r
    bands = ['delta', 'theta', 'alpha', 'beta', 'gamma']; xb = np.arange(len(bands))
    for i, m in enumerate(ms):
        ax[1, 0].bar(xb + i * w, [np.mean([np.nanmean(f[m].get(f'r_{b}', [np.nan])) for f in folds]) for b in bands],
                     w, color=colors[m], label=f'{m} waveform')
        ax[1, 0].plot(xb + i * w, [np.mean([np.nanmean(f[m].get(f'env_{b}', [np.nan])) for f in folds]) for b in bands],
                      'o--', color=colors[m], label=f'{m} envelope')
    ax[1, 0].set_xticks(xb + w * (len(ms) - 1) / 2); ax[1, 0].set_xticklabels(bands)
    ax[1, 0].set_title('band-limited r (bars) / envelope r (dots)'); ax[1, 0].legend(fontsize=7)
    # (d) độ quan sát được dự đoán (chỉ từ input) so với r thật
    xs = np.concatenate([-np.asarray(f['pred_logvar_calib']) for f in folds])
    ys = np.concatenate([np.asarray(f[key]['r']) for f in folds])
    ax[1, 1].scatter(xs, ys, s=6, alpha=.4, color='#264653')
    from scipy.stats import spearmanr
    ax[1, 1].set_title(f'predicted observability vs {key} r (Spearman={spearmanr(xs, ys).correlation:.2f})')
    ax[1, 1].set_xlabel('-log variance (model, from EEG only)'); ax[1, 1].set_ylabel('test r')
    plt.tight_layout(); plt.savefig(f'{FIG_DIR}/{name}_overview.png', dpi=150); plt.show()

In [ ]:
#@title 14. Waveform: thật vs dự đoán (model đã calibrate)
import json, torch, numpy as np, matplotlib.pyplot as plt
from s2i.data.dataset import Subject
from s2i.model.leadformer import LeadFormer, SubjectAdapter
from s2i.inference import predict_trials
from s2i.baselines import LaggedRidge

RUN = f'{RUNS_DIR}/bip_car'   #@param {type:"string"}
SID = 'sub-13'                #@param {type:"string"}
cfg = json.load(open(f'{RUN}/config.json'))
dev = torch.device('cuda')
subj = Subject(f'{DATA_DIR}/{SID}.pt', cfg['target'], cfg['eeg_ref'])
ck = torch.load(f'{RUN}/calib_{SID}.pt', weights_only=False, map_location=dev)
model = LeadFormer(d=cfg['d'], heads=cfg['heads'], enc_layers=cfg['enc_layers'],
                   dec_layers=cfg['dec_layers'], patch=cfg['patch'], dropout=cfg['dropout']).to(dev)
model.load_state_dict(ck['state'])
ad = SubjectAdapter(ck['n_targets'], cfg['d']).to(dev); ad.load_state_dict(ck['adapter'])
tr = ck['test_trials'][:2]
P, T, _ = predict_trials(model, subj, tr, cfg['win'], cfg['win'] // 2, dev, ad)
rg = LaggedRidge().fit([subj.trial(i)[0].float().numpy() for i in ck['cal_trials']],
                       [subj.trial(i)[1].float().numpy() for i in ck['cal_trials']])
R = rg.predict([subj.trial(i)[0].float().numpy() for i in tr])
fold = json.load(open(f'{RUN}/fold_{SID}.json'))
r = np.asarray(fold['calibrated']['r'])
picks = list(np.argsort(-r)[:3]) + [int(np.argsort(-r)[len(r) // 2])]   # 3 tốt nhất + 1 trung vị
t = np.arange(P[0].shape[1]) / 200
fig, axs = plt.subplots(len(picks), 1, figsize=(15, 2.4 * len(picks)), sharex=True)
for a, k in zip(axs, picks):
    a.plot(t, T[0][k], 'k', lw=.8, label='iEEG thật')
    a.plot(t, P[0][k], color='#e76f51', lw=.9, label='LeadFormer')
    a.plot(t, R[0][k], color='#2a9d8f', lw=.7, alpha=.7, label='Ridge')
    a.set_title(f"{subj.tgt_names[k]} | {subj.tgt_region[k][:60]} | {subj.tgt_depth[k]} | "
                f"test r={r[k]:.3f} (ridge {fold['ridge']['r'][k]:.3f})", fontsize=9)
axs[0].legend(loc='upper right'); axs[-1].set_xlabel('s')
plt.tight_layout(); plt.savefig(f'{RUNS_DIR}/figures/waveform_{SID}.png', dpi=150); plt.show()

## Ghi chú diễn giải kết quả
- **Con số chính** là mean r trên **tất cả** contact của **tất cả** subject (cột `calibrated_r` của `bip_car`). Không chọn subset theo kết quả.
- Một tuyên bố "model tốt" chỉ đứng vững khi **calibrated > ridge** (xem `wilcoxon_calibrated_vs_ridge_p` trong `summary.json`) và **zero-shot > 0**.
- So sánh `bip_car` với `mono_orig` cho thấy phần đóng góp của tham chiếu chung trong các con số CAST cũ.
- Mọi file nằm ở `MyDrive/iEEG_S2I/runs/`: `run.log`, `fold_*.json` (metric theo từng contact), `summary.md`, `figures/`.